#### Bronze: ingest road status page
Downloads the NPS Blue Ridge Parkway road status page and stores it **unchanged**, as the raw source for the silver and gold layers. Can run several times a day; only overwrites if the source HTML actually changed.

**Outputs**
-  Download metadata (time, status, fingerprint, file location) plus the HTML as text, so silver can query and parse it with SQL/Spark.: `brp.bronze.road_status_raw`

**Steps**
1. **Settings:** load libraries and define the source URL, storage locations, and today's run date (Eastern time)
2. **Download:** fetch the page; record the exact download time (UTC) and a SHA-256 fingerprint
3. **Gate checks:** stop before saving anything if the download looks wrong (not status 200, too small, or missing the expected headings)
4. **Check for changes:** compare the fingerprint with the most recent download in bronze; if identical, end the run without saving
5. **Save file:** write the untouched HTML to today's folder, named by download time.
6. **Write record:** add this download to the bronze table (MERGE on source + download time, so a rerun can't create a duplicate)
7. **Audit:** confirm the table has no duplicate downloads and no unchanged pages stored back-to-back; fails the job if it does

**Safe to re-run:** an unchanged page is skipped, and rerunning the write step can't duplicate a download.

**If a check fails:** gate failures save nothing, so just rerun or wait for the next scheduled run. For audit failures, delete the bad row by `fetched_at` in bronze.

In [0]:
# ---------------------------------------------------------------------------
# BRONZE INGEST: SETUP
# Settings and imports for the bronze step. This step downloads the NPS
# road status page and saves it exactly as received (no cleaning)
# into the bronze Delta table
# The silver notebook later reads from that table and does the parsing.
# ---------------------------------------------------------------------------


# --- Imports ---

# Standard library (built into Python)
import os                   # building and checking file paths in the volume
import hashlib              # making a fingerprint (hash) of the page, to spot when it changes
from datetime import datetime, timezone   # current date/time; timezone.utc for UTC timestamps
from zoneinfo import ZoneInfo             # real-world time zones like "America/New_York"

# Third-party (installed packages, available on Databricks)
import requests                      # downloading the web page
from pyspark.sql import Row          # building a one-row record to write to the bronze table


# --- Settings ---

# The web page to download. Its HTML is saved unchanged; parsing happens in silver.
SOURCE_URL = "https://www.nps.gov/blri/planyourvisit/roadclosures.htm"


# Bronze Delta table: recording what was downloaded and when.
# Format: catalog.schema.table
BRONZE_TABLE = "brp.bronze.road_status_raw"


# --- Run date ---
# Today's date in the parkway's local time zone (Eastern), not the server's.
# Databricks clusters run on UTC, so a job running at 9 P.M. Eastern would
# otherwise be stamped with tomorrow's date.
#
# Kept in two forms:
#   run_date      -> a real date object; store THIS in the bronze table, so
#                    the column is a DATE type and matches the silver schema
#   run_date_str  -> the same date as text, "2026-11-13"; use THIS in any file
#                    names, since file paths need text
run_date = datetime.now(ZoneInfo("America/New_York")).date()
run_date_str = run_date.isoformat()
print("Run date:", run_date_str)

In [0]:
# ---------------------------------------------------------------------------
# BRONZE INGEST: DOWNLOAD THE PAGE
# Fetch the NPS road status page and keep it exactly as received. Nothing is
# cleaned or parsed here; that happens in silver. Keeping the original means
# that if a parsing rule is ever wrong, silver can be rebuilt from bronze.
# ---------------------------------------------------------------------------


# --- Step 1: Request the page ---
response = requests.get(
    SOURCE_URL,

    # Give up after 30 seconds if NPS doesn't answer, so a slow or down site
    # fails the job quickly instead of leaving it hanging.
    timeout=30,

    # Identify this script to the website. Without a User-Agent, requests
    # sends a generic one ("python-requests/..."), which some sites block.
    # A descriptive name is also polite: the site owner can see who's calling.
    headers={"User-Agent": "brp-closures-portfolio-project"},
)

# Stop the job if NPS returned an error page (e.g. 404 Not Found, 403
# Forbidden, 500 Server Error). Without this, the error page's HTML would be
# saved to bronze as if it were the real road status page.
response.raise_for_status()


# --- Step 2: Record what was downloaded, for auditing ---

# The percise timestamp of the download, in UTC. UTC avoids any ambiguity
# around time zones and daylight saving changes. (unlike run_date, from the setup
# cell, which is the Eastern calendar date.)
fetched_at = datetime.now(timezone.utc)

# The page exactly as received, as raw bytes. .content gives the bytes
# untouched; .text would decode them into a string first, which can change
# characters if the encoding is guessed wrong.
raw_bytes = response.content

# A SHA-256 fingerprint of those bytes: a 64-character code that's identical
# for identical content and completely different if even one character
# changes. Useful for spotting days when the page didn't change (same fingerprint as yesterday)
sha256 = hashlib.sha256(raw_bytes).hexdigest()


# --- Step 3: Print a short summary for the run log ---
print("Fetch timestamp:", fetched_at)
print("Status:", response.status_code)            # should be 200 (OK); errors stopped above
print("Size:", len(raw_bytes), "bytes")           # a sudden big change may mean a layout change
print("Fingerprint:", sha256[:16], "...")         # first 16 characters are enough to compare by eye

In [0]:
# ---------------------------------------------------------------------------
# BRONZE GATE: IS THIS DOWNLOAD WORTH SAVING?
# Checks the new download BEFORE anything is written. If any check fails,
# the job stops here, and nothing is saved to the bronze table.
# ---------------------------------------------------------------------------

page_text = response.text

# Gate 1: The site said OK. (raise_for_status already stops 4xx/5xx errors;
# this also catches odd replies like 204 "No Content".)
assert response.status_code == 200, f"Gate 1: Unexpected status {response.status_code}"

# Gate 2: The page isn't suspiciously small. A real road status page is
# tens of thousands of characters; an error or "maintenance" page is tiny.
# Adjust the number as needed but page size should be around 60_000 bytes.
assert len(raw_bytes) > 40_000, f"Gate 2: Page only {len(raw_bytes)} bytes: error page?"

# Gate 3: The page contains what silver needs to parse. If NPS redesigns the
# page, these phrases disappear, so it's better to stop now than to store a page
# that silver can't read.
assert "Sections of Parkway" in page_text, "Gate 3: State table headings missing: page layout changed?"
assert "Road status as of" in page_text, "Gate 3: 'Road status as of' line missing: page layout changed?"

print("Gate checks passed")

In [0]:
# ---------------------------------------------------------------------------
# BRONZE INGEST: SKIP IF UNCHANGED
# Compare this download with the most recent one stored in bronze. If the
# fingerprint matches, nothing new needs saving.
# The result is kept in `page_changed` (True/False) so the save and write
# cells below can check it too, which keeps manual cell-by-cell runs safe.
# ---------------------------------------------------------------------------

# Start with "nothing to compare against". This stays None on the very first
# run, when the bronze table doesn't exist yet.
last = None

if spark.catalog.tableExists(BRONZE_TABLE):
    last = spark.sql(f"""
        SELECT sha256
        FROM {BRONZE_TABLE}
        ORDER BY fetched_at DESC      -- newest download first
        LIMIT 1                       -- there should be only one
    """).first()

# True  -> no previous download, or a different fingerprint: save it
# False -> same fingerprint as the last download: skip it
page_changed = not (last is not None and last["sha256"] == sha256)

if page_changed:
    print("Page is new or changed; saving.")
else:
    print("Page unchanged since last download; nothing will be saved.")
    dbutils.notebook.exit("unchanged")   # in a job or Run all, the run ends here

In [0]:
# ---------------------------------------------------------------------------
# BRONZE INGEST: RECORD THIS DOWNLOAD IN THE BRONZE TABLE
# Add one row describing this download to the bronze Delta table.
# The silver notebook reads the row from here.
# ---------------------------------------------------------------------------


# --- Build the record for this download ---
record = Row(
    source="road_status_page",         # which source this is (room for more sources later)
    url=SOURCE_URL,                    # the exact page that was downloaded
    run_date=run_date,                 # Eastern calendar date (a real date), for grouping by day
    fetched_at=fetched_at,             # exact UTC moment of the download: unique per run
    http_status=response.status_code,  # the site's response code (200 = OK)
    sha256=sha256,                     # fingerprint of the page, used to skip unchanged downloads
    raw_html=response.text,            # the page as text, ready for silver to parse
)

# Turn the record into a one-row Spark DataFrame so it can be written to a table.
df = spark.createDataFrame([record])
# MERGE needs a named view to read from, so register the DataFrame
# under the temporary name "incoming".
df.createOrReplaceTempView("incoming")


# --- Replace or add the row for this source ---
# Match on source only, so there's never more than one row per source:
#   - Row exists, page changed  -> replace it with this download
#   - Row exists, page the same -> leave it alone (normally the skip cell
#                                  already stopped the run; this is a safety net)
#   - No row yet (first run)    -> insert it
spark.sql(f"""
    MERGE INTO {BRONZE_TABLE} AS t
    USING incoming AS s
    ON t.source = s.source
    WHEN MATCHED AND t.sha256 <> s.sha256 THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
print("Bronze now holds the latest page for", record.source)

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- BRONZE REVIEW: WHAT'S BEEN DOWNLOADED
-- Run to confirm the latest download landed and looks reasonable.
--
-- What to look for:
--   - http_status should be 200.
--   - html_length should be roughly 60_000 bytes. A sudden big
--     drop (or near 0) usually means an error page or a site redesign.
-- ---------------------------------------------------------------------------
SELECT
  run_date,                                         -- pipeline day (America/New_York); several rows can share one
  fetched_at,                                       -- exact download time (UTC)
  from_utc_timestamp(fetched_at, 'America/New_York')
    AS fetched_at_eastern,                          -- same moment in Eastern time, easier to read
  http_status,                                      -- server reply code; 200 means OK
  left(sha256, 16) AS sha256_short,                 -- first 16 characters of the fingerprint; enough to compare by eye
  length(raw_html) AS html_length                   -- page size in characters
FROM brp.bronze.road_status_raw
